# Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy as sp
import statsmodels as sm
import sklearn as sk
import os
import re
import time
from tqdm import tqdm

In [ ]:
%store -r static_reference_data
%store -r restaurant_sales_data

In [ ]:
before_after_details = static_reference_data[0]
customers = static_reference_data[1]
items_tagged = static_reference_data[2]
locations = static_reference_data[3]

Static reference data exploration

In [ ]:
items_tagged.columns

In [ ]:
items_tagged['item_type'].value_counts()

In [ ]:
items_tagged['dish_category'].value_counts()

In [ ]:
items_tagged['ingredients'].value_counts()

In [ ]:
items_tagged['brand'].value_counts()

In [ ]:
plt.bar(items_tagged['is_plant_based'].value_counts().index, items_tagged['is_plant_based'].value_counts())
plt.title("Is Plant-Based?")
plt.show()

In [ ]:
plt.bar(items_tagged['item_type'].value_counts().index, items_tagged['item_type'].value_counts())
plt.title("Item Types")
plt.show()

In [ ]:
plt.bar(items_tagged['dish_category'].value_counts()[:20].index, items_tagged['dish_category'].value_counts()[:20])
plt.title("Dish Categories")
plt.xticks(rotation = 75)
plt.show()

In [ ]:
items_tagged_noalc = items_tagged[items_tagged['dish_category'] != 'Alcohol']
plant_based = items_tagged_noalc.loc[items_tagged_noalc['is_plant_based'] == 'yes',:]

In [ ]:
plt.bar(items_tagged_noalc['is_plant_based'].value_counts().index, items_tagged_noalc['is_plant_based'].value_counts(), color="red")
plt.title("Is Plant-Based?")
plt.show()

In [ ]:
plt.bar(items_tagged_noalc['item_type'].value_counts().index, items_tagged_noalc['item_type'].value_counts(), color="red")
plt.title("Item Types")
plt.show()

In [ ]:
plt.bar(items_tagged_noalc['dish_category'].value_counts()[:20].index, items_tagged_noalc['dish_category'].value_counts()[:20], color="red")
plt.title("Dish Categories")
plt.xticks(rotation = 75)
plt.show()

In [ ]:
plt.bar(plant_based['item_type'].value_counts().index, plant_based['item_type'].value_counts(), color="green")
plt.title("Item Types")
plt.show()

In [ ]:
plt.bar(plant_based['dish_category'].value_counts()[:20].index, plant_based['dish_category'].value_counts()[:20], color="green")
plt.title("Dish Categories")
plt.xticks(rotation = 75)
plt.show()

In [ ]:
def df_like_dict(df, col, key):
    return df.loc[df[col].str.lower() == key, :]

In [ ]:
plant = df_like_dict(items_tagged, 'is_plant_based', 'yes')
print(plant.shape, plant.loc[plant['item_type'] != 'drink',:].shape)

Time series exploration

In [ ]:
location_ids = list(restaurant_sales_data.keys())

In [ ]:
list(restaurant_sales_data.values())[1]['item_price'].value_counts()

First Restaurant

In [ ]:
list_of_foods_rest_1 = restaurant_sales_data[location_ids[0]].loc[:,['item_name']].head(100)

In [ ]:
plt.bar(restaurant_sales_data[location_ids[0]]['item_name'].value_counts().iloc[:20].index, restaurant_sales_data[location_ids[0]]['item_name'].value_counts().iloc[:20])
plt.xticks(rotation=75)
plt.show()

In [ ]:
pd.set_option('display.max_rows', 100)

In [ ]:
merged1 = pd.merge(restaurant_sales_data[location_ids[0]].reset_index(), items_tagged.copy()[['item_name','is_plant_based']], left_on='item_name', right_on='item_name', how='left')
merged1.set_index('created_at', drop=True, inplace=True)

In [ ]:
before_after_details

In [ ]:
df_like_dict(merged1, 'is_plant_based', 'yes').head()

In [ ]:
df_like_dict(df_like_dict(merged1, 'is_plant_based', 'yes'), 'item_name', 'impossible burger')

All restaurants

In [ ]:
merged_data = []

In [ ]:
for location_id, df in restaurant_sales_data.items():
    merged = pd.merge(df.reset_index(), items_tagged.copy()[['item_name','is_plant_based']], left_on='item_name', right_on='item_name', how='left')
    merged.set_index('created_at', drop=True, inplace=True)
    plt.bar(df['item_name'].value_counts().iloc[:20].index.str.slice(0,20), df['item_name'].value_counts().iloc[:20])
    plt.xticks(rotation=75)
    plt.show()

In [ ]:
# Step 2: Filter Data by Item Name
def filter_data_by_item(data, item_name):
    return data[data['item_name'] == item_name]

# Step 3: Plot Time Series
def plot_time_series(data, item_name):
    filtered_data = filter_data_by_item(data, item_name)
    filtered_data['timestamp'] = pd.to_datetime(filtered_data['timestamp'])
    plt.figure(figsize=(10, 6))
    plt.plot(filtered_data['timestamp'], filtered_data['sale_amount'])
    plt.title(f'Time Series of Sales for {item_name}')
    plt.xlabel('Timestamp')
    plt.ylabel('Sale Amount')
    plt.grid(True)
    plt.show()

# Load your data (replace 'your_sales_data.csv' with your actual file path)
sales_data = load_data('your_sales_data.csv')

# Step 4: User Input
item_to_plot = input("Enter the name of the item to plot: ")
plot_time_series(sales_data, item_to_plot)